# Simulating Match with Trained Players

This notebook shows two different way to perform to simulate a match

## 1. Running `unity_inference.py` (RECOMMENDED)
Launches the Unity executable as a subprocess, passes ONNX models via command-line flags (`--mlagents-override-model-directory`), and lets Unity handle inference internally. Does not depend on ML-Agents.

Use `unity_inference` when you want fast, headless batch simulations (e.g. tournaments, automated evaluation pipelines) — it's lower overhead and self-terminating.


## 2. Running `mlagents_inference.py` 
Connects to Unity via the ML-Agents Python API, runs ONNX inference itself using `onnxruntime`, and sends actions back to Unity step-by-step. Depends on ML-Agents.

Use `mlagents_inference` when you need rich step-level telemetry (cumulative rewards, per-agent outcomes, goal events), want to inspect or modify behavior mid-run, or are debugging a model's decision-making.


## Summary Table of Differences

Here is the data converted into a clean, scannable Markdown table:

| Feature | `unity_inference` | `mlagents_inference` |
| --- | --- | --- |
| **Inference runs in** | Unity (C# side) | Python (onnxruntime) |
| **Integration** | Subprocess + log parsing | Live ML-Agents env loop |
| **Termination** | Auto (episode/timeout limits) | Manual (Ctrl-C) |
| **Score extraction** | Log file regex parsing | Direct reward signals |
| **Observability** | Low — post-hoc log parsing | High — per-step rewards, per-agent events |
| **Speed** | Faster (no Python step loop overhead) | Slower (every step round-trips through Python) |
| **Video capture** | Built-in (`--save-video`) | Not supported |
| **Robustness** | Depends on Unity log format staying stable | Depends on ML-Agents Python API |



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import unity_inference
import mlagents_inference
from inference_utils import print_score

## Simulating Match with Unity Executable

Press spacebar during simulation to cycle through camera modes.

In [ ]:
results_dir = 'results'

models = {
    "teamLeft": "models/example-model-A.onnx",
    "teamRight": "models/example-model-B.onnx"
}

In [38]:
# Simulate gameplay and save video

UNITY_ENV = 'env/MacOS/football-inference'

result = unity_inference.simulate_match(
    models = models,
    env_path=UNITY_ENV,
    results_dir=results_dir,
    episode_limit=5,
    timeout_limit=20,
    save_video=True
)

print_score(result)
print(f"Video saved in {result.result_path}")

Final Score: 0 -- 1!
Video saved in /Users/ilari/Games/Unity/chibi-football-lab/results/20260626-153308


In [35]:
# Evaluate relative performance of the models efficiently through multiple fields

UNITY_ENV = 'env/MacOS/football-evaluation'

result = unity_inference.simulate_match(
    models = models,
    env_path=UNITY_ENV,
    results_dir=results_dir,
    timeout_limit=20
)

print_score(result)

Final Score: 6 -- 8!


In [ ]:
# Test custom training environment, which uses self-play and thus accepts only a single model as input

UNITY_ENV = 'env/MacOS/football-training'

# Model key must be named "SoccerTwos" to match behavior name in executable
training_model = {
    "SoccerTwos": "models/example-model-A.onnx"
}

result = unity_inference.simulate_match(
    models = training_model,
    env_path=UNITY_ENV,
    results_dir=results_dir,
    timeout_limit=20
)

print_score(result)

Final Score: 3 -- 5!


## Simulating a Match via ML-Agents

Alternative way to simulate matches.
The module `mlagents_inference.py` uses `mlagents` which provides some additional ways to inspect the environment details during the simulation.

In [ ]:
mlagents_inference.simulate_match(
    env_path=UNITY_ENV,
    left_model_path=models['teamLeft'],
    right_model_path=models['teamRight'],
)
